<a href="https://colab.research.google.com/github/RushitPatel/deepfack_project/blob/main/notebooks/deepfake_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os

# 1. Clone the repository
repo_url = "https://github.com/RushitPatel/deepfack_project.git"
repo_name = "deepfack_project"

if not os.path.exists(repo_name):
    !git clone {repo_url}
    %cd {repo_name}
else:
    %cd {repo_name}
    !git pull

# 2. Install requirements and the package itself
!pip install -r requirements.txt
!pip install -e .

Cloning into 'deepfack_project'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 105 (delta 52), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 76.48 KiB | 5.10 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/content/deepfack_project
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.9 MB/s eta 0:00:00
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Obtaining file:///content/deepfack_project
ERROR: file:///content/deepfack_project does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [ ]:
import sys
!{sys.executable} -m pip install numpy==1.26.4 --force-reinstall
!{sys.executable} -m pip install scipy --force-reinstall
!{sys.executable} -m pip install scikit-learn --force-reinstall
!{sys.executable} -m pip install facenet_pytorch
# =============================================================================
# deepfake_detection.py
# =============================================================================
#   Multi‑modal deep‑fake detector (EfficientNet‑B4 + ViT‑Small + ResNet‑18)
#   All logic lives in this single file so that any Colab notebook can do:
#       import deepfake_detection as dfd
#   and have immediate access to:c
#       - DeepFakeDataset  (video + audio preprocess)
#       - DeepFakeDetector (model + forward)
#       - train(...), evaluate_on_loader(...), run_inference(...)
#       - grad_cam_video(...)
#   -------------------------------------------------------------------------
#   📌 Author tags (used for the Contribution Statement):
#       • Isha Luhar      – config, reproducibility utils, dataset class
#       • Nishtha Solanki – model definitions, Grad‑CAM helper
#       • OM Mistry       – training loop, checkpointing, metric logging
#       • Rushitkumar Patel – evaluation helpers, fairness audit, latency code
#   -------------------------------------------------------------------------
#   📚 References to the papers from the Part‑1 design are in the doc‑strings.
# =============================================================================

# ------------------------------------------------------------
# 0️⃣  Imports & global configuration
# ------------------------------------------------------------
import os
import random
import time
import json
import yaml
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import timm
from torchvision import models
import librosa
import cv2
from facenet_pytorch import MTCNN
from tqdm.auto import tqdm

from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from torchcam.methods import SmoothGradCAMpp
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1️⃣  Global constants (edit only DATA_ROOT in the notebook)
# ------------------------------------------------------------
SEED   = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Path where the three splits (train/val/test) live – set in notebook 01
DATA_ROOT = Path("/content/drive/MyDrive/deepfake_data")   # <‑‑ will be overwritten

def set_seed(seed: int = SEED) -> None:
    """Make the whole pipeline deterministic (as far as possible)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

def timer(func):
    """Decorator – returns (output, elapsed_ms)."""
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        out   = func(*args, **kwargs)
        elapsed_ms = (time.perf_counter() - start) * 1000
        return out, elapsed_ms
    return wrapper

# ------------------------------------------------------------
# 2️⃣  Helper – simple CSV logger (append‑only, reproducible)
# ------------------------------------------------------------
def log_to_csv(csv_path: Path, row_dict: dict) -> None:
    """Append a dict as a CSV row (creates file with header on first call)."""
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not csv_path.is_file()
    with open(csv_path, "a", newline="") as f:
        import csv
        writer = csv.DictWriter(f, fieldnames=row_dict.keys())
        if write_header:
            writer.writeheader()
        writer.writerow(row_dict)

# ------------------------------------------------------------
# 3️⃣  Dataset – video + audio + label
# ------------------------------------------------------------
# === [Author: Isha Luhar] -------------------------------------------------
class DeepFakeDataset(Dataset):
    """
    Expected directory layout (all under DATA_ROOT):
        <split>/videos/*.mp4   (or .avi)
        <split>/audio/*.wav    (same stem as video)
        <split>/labels.csv    (columns: file,label)

    Returns a dict:
        {
            "video": Tensor[T,3,224,224],
            "audio": Tensor[1,128,T'],
            "label": 0/1,
            "metadata": {"video_path": str, "audio_path": str}
        }
    """
    def __init__(self,
                 split: str = "train",
                 target_fps: int = 10,
                 n_mels: int = 128,
                 frames_per_clip: int = 16):
        self.split_dir      = DATA_ROOT / split
        self.video_dir      = self.split_dir / "videos"
        self.audio_dir      = self.split_dir / "audio"
        self.labels_df      = pd.read_csv(self.split_dir / "labels.csv")
        self.target_fps     = target_fps
        self.n_mels         = n_mels
        self.frames_per_clip = frames_per_clip
        self.mtcnn           = MTCNN(select_largest=False,
                                    keep_all=False,
                                    device='cpu')   # CPU is fine for preprocessing

        # Build a list of (video_path, audio_path, label) tuples
        self.samples = []
        for _, row in self.labels_df.iterrows():
            stem  = row["file"]
            label = int(row["label"])
            vid_path = self.video_dir / f"{stem}.mp4"
            if not vid_path.is_file():
                vid_path = self.video_dir / f"{stem}.avi"
            aud_path = self.audio_dir / f"{stem}.wav"
            if vid_path.is_file() and aud_path.is_file():
                self.samples.append((vid_path, aud_path, label))

    def __len__(self) -> int:
        return len(self.samples)

    # -----------------------------------------------------------------
    # Private helpers for a single sample
    # -----------------------------------------------------------------
    def _load_video(self, video_path: Path) -> torch.Tensor:
        """Extract `frames_per_clip` face‑cropped frames (T,3,224,224)."""
        cap = cv2.VideoCapture(str(video_path))
        orig_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        step = max(int(orig_fps // self.target_fps), 1)

        frames = []
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if idx % step == 0:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                # ---------- face detection ----------
                box, _ = self.mtcnn.detect(rgb)
                if box is not None:
                    x1, y1, x2, y2 = map(int, box[0])
                    face = rgb[y1:y2, x1:x2]
                else:
                    h, w, _ = rgb.shape
                    sz = min(h, w) // 2
                    cx, cy = w // 2, h // 2
                    face = rgb[cy - sz:cy + sz, cx - sz:cx + sz]

                face = cv2.resize(face, (224, 224))
                face = torch.from_numpy(face).permute(2, 0, 1).float() / 255.0
                frames.append(face)
            idx += 1
        cap.release()

        # Pad / truncate to the required number of frames
        T = self.frames_per_clip
        if len(frames) < T:
            pad = [torch.zeros_like(frames[0]) for _ in range(T - len(frames))]
            frames.extend(pad)
        elif len(frames) > T:
            frames = frames[:T]
        return torch.stack(frames)   # (T,3,224,224)

    def _load_audio(self, audio_path: Path) -> torch.Tensor:
        """Log‑mel spectrogram (1, n_mels, time)."""
        y, sr = librosa.load(str(audio_path), sr=16000)
        mel = librosa.feature.melspectrogram(
            y,
            sr=sr,
            n_fft=512,
            hop_length=160,          # 10 ms frames
            n_mels=self.n_mels,
        )
        log_mel = np.log1p(mel)
        return torch.from_numpy(log_mel).unsqueeze(0).float()   # (1,n_mels,T)

    def __getitem__(self, idx):
        video_path, audio_path, label = self.samples[idx]
        video = self._load_video(video_path)      # (T,3,224,224)
        audio = self._load_audio(audio_path)      # (1,n_mels,T')
        return {
            "video": video,
            "audio": audio,
            "label": torch.tensor(label, dtype=torch.float32),
            "metadata": {
                "video_path": str(video_path),
                "audio_path": str(audio_path)
            }
        }

def deepfake_collate_fn(batch):
    """Collate function for DataLoader – stacks tensors."""
    videos = torch.stack([b["video"] for b in batch])      # (B,T,3,224,224)
    audios = torch.stack([b["audio"] for b in batch])      # (B,1,n_mels,T')
    labels = torch.stack([b["label"] for b in batch])
    meta   = [b["metadata"] for b in batch]
    return {"video": videos, "audio": audios,
            "label": labels, "metadata": meta}

# ------------------------------------------------------------
# 4️⃣  Model definitions
# ------------------------------------------------------------
# === [Author: Nishtha Solanki] -----------------------------------------
class VideoPipeline(nn.Module):
    """EfficientNet‑B4 (spatial) → 4‑layer ViT‑Small (temporal) → 1‑dim logit."""
    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model(
            "efficientnet_b4",
            pretrained=pretrained,
            num_classes=0            # feature extractor, no classifier head
        )
        enc_layer = nn.TransformerEncoderLayer(
            d_model=1792,
            nhead=8,
            batch_first=True,
            dim_feedforward=1792 * 4,
            dropout=0.1,
            activation="gelu"
        )
        self.temporal = nn.TransformerEncoder(enc_layer, num_layers=4)
        self.fc = nn.Linear(1792, 1)   # final video logit

    def forward(self, x):
        """
        x: (B, T, C, H, W)
        Returns: (B,) raw logits.
        """
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)           # flatten temporal dim
        feats = self.backbone(x)              # (B*T, 1792)
        feats = feats.view(B, T, -1)          # (B, T, 1792)
        feats = self.temporal(feats)          # (B, T, 1792)
        last = feats[:, -1, :]                # use the last token
        logit = self.fc(last).squeeze(-1)     # (B,)
        return logit


class AudioPipeline(nn.Module):
    """ResNet‑18 adapted for 1‑channel Mel‑spectrogram → 1‑dim logit."""
    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.backbone = models.resnet18(pretrained=pretrained)
        self.backbone.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False
        )
        self.backbone.fc = nn.Linear(512, 1)

    def forward(self, x):
        """
        x: (B, 1, n_mels, T')
        Returns: (B,) raw logits.
        """
        logit = self.backbone(x).squeeze(-1)   # (B,)
        return logit


class MetaLearner(nn.Module):
    """Linear meta‑learner that fuses the two modality logits."""
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(2, 1)   # (v_logit, a_logit) → fused logit

    def forward(self, v_logit, a_logit):
        xy = torch.stack([v_logit, a_logit], dim=1)   # (B,2)
        fused = self.fc(xy).squeeze(-1)               # (B,)
        return fused


class DeepFakeDetector(nn.Module):
    """Full end‑to‑end multi‑modal detector."""
    def __init__(self):
        super().__init__()
        self.video_branch = VideoPipeline()
        self.audio_branch = AudioPipeline()
        self.meta_learner = MetaLearner()

    def forward(self, video, audio):
        """
        video: (B,T,3,224,224)
        audio: (B,1,n_mels,T')
        Returns: (fused_prob, video_prob, audio_prob) – all are sigmoids.
        """
        v_logit   = self.video_branch(video)      # (B,)
        a_logit   = self.audio_branch(audio)      # (B,)
        fused_logit = self.meta_learner(v_logit, a_logit)   # (B,)

        v_prob   = torch.sigmoid(v_logit)
        a_prob   = torch.sigmoid(a_logit)
        fused_prob = torch.sigmoid(fused_logit)

        return fused_prob, v_prob, a_prob

# ------------------------------------------------------------
# 5️⃣  Grad‑CAM helper (video branch only)
# ------------------------------------------------------------
# === [Author: Nishtha Solanki] -----------------------------------------
def grad_cam_video(model: DeepFakeDetector, video_tensor: torch.Tensor):
    """
    Returns a NumPy heat‑map (H×W) for the frame that the ViT attends to most.
    The function expects a *batch* of video frames (B,T,3,224,224);
    for inference we normally use B=1.
    """
    # Grab EfficientNet backbone (inside the video branch)
    backbone = model.video_branch.backbone
    cam_extractor = SmoothGradCAMpp(backbone, target_layer='blocks[-1]')

    B, T, C, H, W = video_tensor.shape
    flat = video_tensor.view(B * T, C, H, W)

    # Forward through backbone (required for the hook)
    _ = backbone(flat)
    cams = cam_extractor(class_idx=None)   # list length B*T, each (H,W)

    cams = torch.stack(cams).view(B, T, H, W)   # (B,T,H,W)

    # Determine the most‑attended frame via the norm of the last token
    with torch.no_grad():
        feats = backbone(flat)                       # (B*T,1792)
        feats = feats.view(B, T, -1)                 # (B,T,1792)
        temporal = model.video_branch.temporal(feats) # (B,T,1792)
        last_token = temporal[:, -1, :]                # (B,1792)
        token_norm = torch.norm(last_token, dim=1)   # (B,)

        # For batch‑size 1 we just pick the single frame
        best_idx = torch.argmax(token_norm).item() if B > 1 else 0

    cam_np = cams[0, best_idx].cpu().numpy()
    cam_np = (cam_np - cam_np.min()) / (cam_np.max() - cam_np.min() + 1e-8)
    return cam_np

# ------------------------------------------------------------
# 6️⃣  Training utilities (loss, metrics, epoch loops, checkpointing)
# ------------------------------------------------------------
# === [Author: OM Mistry] ------------------------------------------------
def compute_metrics(probs: np.ndarray, labels: np.ndarray):
    """Return dict with accuracy, ROC‑AUC, and FPR@0.5."""
    preds = (probs >= 0.5).astype(int)
    acc   = accuracy_score(labels, preds)
    auc   = roc_auc_score(labels, probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    return {"accuracy": acc, "auc": auc, "fpr": fpr,
            "tp": tp, "tn": tn, "fp": fp, "fn": fn}


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    epoch_loss = 0.0
    all_probs  = []
    all_labels = []

    for batch in loader:
        video = batch["video"].to(DEVICE)
        audio = batch["audio"].to(DEVICE)
        label = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        fused_prob, _, _ = model(video, audio)                # (B,)
        loss = criterion(torch.logit(fused_prob), label)     # BCEWithLogits
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * video.size(0)
        all_probs.extend(fused_prob.detach().cpu().numpy())
        all_labels.extend(label.cpu().numpy())

    avg_loss = epoch_loss / len(loader.dataset)
    metrics = compute_metrics(np.array(all_probs), np.array(all_labels))
    metrics["loss"] = avg_loss
    return metrics


def validate_one_epoch(model, loader, criterion):
    model.eval()
    epoch_loss = 0.0
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            video = batch["video"].to(DEVICE)
            audio = batch["audio"].to(DEVICE)
            label = batch["label"].to(DEVICE)

            fused_prob, _, _ = model(video, audio)
            loss = criterion(torch.logit(fused_prob), label)

            epoch_loss += loss.item() * video.size(0)
            all_probs.extend(fused_prob.cpu().numpy())
            all_labels.extend(label.cpu().numpy())

    avg_loss = epoch_loss / len(loader.dataset)
    metrics = compute_metrics(np.array(all_probs), np.array(all_labels))
    metrics["loss"] = avg_loss
    return metrics


def train(model,
          train_loader,
          val_loader,
          num_epochs: int = 25,
          lr: float = 1e-4,
          weight_decay: float = 1e-5,
          lr_step: int = 10,
          lr_gamma: float = 0.5,
          checkpoint_path: str = "best_model.pth",
          log_csv: str = "training_log.csv"):
    """
    High‑level training loop.
    * Saves the checkpoint with the highest validation AUC.
    * Appends a CSV row after every epoch (reproducibility trace).
    """
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr,
                            weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer,
                                          step_size=lr_step,
                                          gamma=lr_gamma)

    best_auc = 0.0
    for epoch in range(1, num_epochs + 1):
        train_metrics = train_one_epoch(model, train_loader,
                                       criterion, optimizer)
        val_metrics   = validate_one_epoch(model, val_loader,
                                          criterion)

        scheduler.step()

        # Log to CSV (each epoch is a row)
        log_row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc":  train_metrics["accuracy"],
            "train_auc":  train_metrics["auc"],
            "train_fpr":  train_metrics["fpr"],
            "val_loss":   val_metrics["loss"],
            "val_acc":    val_metrics["accuracy"],
            "val_auc":    val_metrics["auc"],
            "val_fpr":    val_metrics["fpr"],
        }
        log_to_csv(Path(log_csv), log_row)

        # Human‑readable printing
        print(f"\nEpoch {epoch:02d} | "
              f"train loss={train_metrics['loss']:.4f} "
              f"acc={train_metrics['accuracy']:.3f} "
              f"auc={train_metrics['auc']:.4f} "
              f"fpr={train_metrics['fpr']:.4f} | "
              f"val loss={val_metrics['loss']:.4f} "
              f"acc={val_metrics['accuracy']:.3f} "
              f"auc={val_metrics['auc']:.4f} "
              f"fpr={val_metrics['fpr']:.4f}")

        # Save best‑validation‑AUC checkpoint
        if val_metrics["auc"] > best_auc:
            best_auc = val_metrics["auc"]
            torch.save(model.state_dict(), checkpoint_path)
            print(f"*** Saved new best model (val AUC = {best_auc:.4f}) ***")

    print("\nTraining finished. Best validation AUC =", best_auc)
    return model, best_auc

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26

ModuleNotFoundError: No module named 'numpy.char'